# 01 — Whole-Brain Visual System (mcHH Network)

Build a multi-compartment HH network covering **all 95K visual neurons**
(both hemispheres) from the FAFB connectome.

### Timing estimates
| Simulation | Steps | Est. time |
|-----------|-------|----------|
| 200ms flash | 2,000 | ~3 min |
| 500ms flash | 5,000 | ~7 min |
| 12-dir motion (300ms each) | 36,000 | ~55 min |

In [ ]:
import sys, time, logging
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, '../..')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from neuro_framework.models.fafb_mc_network import FAFBMCNetwork

## 1. Build network

In [ ]:
DATA_DIR  = '../../mcHH/data/visual_whole_brain'
SWC_PATH  = '../../mcHH/data/generic_2comp.swc'
ION_RULES = '../data/fafb_ion_channel_rules.csv'
SYN_RULES = '../data/fafb_synapse_rules.csv'

t0 = time.time()
net = FAFBMCNetwork.from_preprocessed(
    data_dir=DATA_DIR, swc_path=SWC_PATH,
    ion_rules_path=ION_RULES, syn_rules_path=SYN_RULES,
    ncomp=2, min_syn_count=3, dt=0.1,
)
print(f'Build: {time.time()-t0:.1f}s')
print(net)

In [ ]:
tc = net.type_counts()
print(f'Total types: {len(tc)}, Photoreceptors: {net.input_mask.sum().item()}')

top = sorted(tc.items(), key=lambda x: -x[1])[:25]
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(range(len(top)), [v for _,v in top], color='steelblue')
ax.set_yticks(range(len(top)))
ax.set_yticklabels([k for k,_ in top])
ax.invert_yaxis()
ax.set_xlabel('Count'); ax.set_title('Top-25 neuron types (whole brain)')
plt.tight_layout(); plt.show()

## 2. Subsystem distribution

In [ ]:
neurons_df = pd.read_csv(DATA_DIR + '/neurons.csv')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ss = neurons_df['subsystem'].value_counts()
axes[0].pie(ss.values, labels=ss.index, autopct='%1.0f%%')
axes[0].set_title('Neurons by subsystem')

side = neurons_df['side'].value_counts()
axes[1].bar(side.index, side.values, color=['steelblue','coral'])
axes[1].set_ylabel('Count'); axes[1].set_title('Left vs Right')
plt.tight_layout(); plt.show()

## 3. Flash simulation (200ms)

In [ ]:
DT = 0.1
T_PRE, T_ON, T_OFF = 50.0, 100.0, 50.0
T_steps = int((T_PRE + T_ON + T_OFF) / DT)
I_AMP = 30.0

n_photo = net.input_mask.sum().item()
x = torch.zeros(1, T_steps, n_photo)
t_on_s = int(T_PRE / DT)
t_on_e = int((T_PRE + T_ON) / DT)
x[:, t_on_s:t_on_e, :] = I_AMP

with torch.no_grad():
    V = net(x, dt=DT, show_progress=True)

print(f'V: [{V.min():.1f}, {V.max():.1f}], NaN={torch.isnan(V).any().item()}')

In [ ]:
t_ms = np.arange(V.shape[1]) * DT
pathway = ['R1-6', 'L1', 'L3', 'L5', 'Mi1', 'Mi9', 'Tm3',
           'T4a', 'T4b', 'T5a', 'T5b', 'C2', 'C3', 'T1', 'T2', 'T3']

fig, axes = plt.subplots(4, 4, figsize=(16, 12), sharex=True)
for i, ct in enumerate(pathway):
    ax = axes.flat[i]
    idx = net.get_indices_by_type(ct)
    if not idx: ax.set_title(f'{ct} (0)'); continue
    v_m = V[0,:,idx].mean(dim=1).numpy()
    ax.plot(t_ms, v_m, 'k', lw=1.5)
    ax.axvspan(T_PRE, T_PRE+T_ON, color='gold', alpha=0.15)
    ax.set_title(f'{ct} (n={len(idx)})')
    ax.set_ylabel('mV')
fig.suptitle('Flash responses — whole visual brain', fontsize=14)
plt.tight_layout(); plt.show()

## 4. FRI + left vs right comparison

In [ ]:
v_base_t = V[0, :t_on_s, :].mean(dim=0)
r_on_t  = V[0, t_on_s:t_on_e, :].mean(dim=0) - v_base_t
r_off_t = V[0, t_on_e:, :].mean(dim=0) - v_base_t
fri_t = (r_on_t - r_off_t) / (r_on_t.abs() + r_off_t.abs() + 1e-8)

all_ids = np.sort(neurons_df['root_id'].unique())
id_to_idx = {int(r): i for i, r in enumerate(all_ids)}
neurons_df['idx'] = neurons_df['root_id'].map(id_to_idx)

print(f'{"Type":>8s}  {"n_L":>5s} {"FRI_L":>7s}  {"n_R":>5s} {"FRI_R":>7s}')
for ct in ['R1-6','L1','L3','Mi1','Mi9','Tm3','T4a','T5a']:
    for side in ['left', 'right']:
        sub = neurons_df[(neurons_df['type']==ct) & (neurons_df['side']==side)]
        il = sub['idx'].dropna().astype(int).tolist()
        if not il: continue
        it = torch.tensor(il)
        if side == 'left':
            n_l, f_l = len(il), fri_t[it].mean().item()
        else:
            n_r, f_r = len(il), fri_t[it].mean().item()
    if 'n_l' in dir() and 'n_r' in dir():
        print(f'{ct:>8s}  {n_l:5d} {f_l:+.3f}  {n_r:5d} {f_r:+.3f}')
    del n_l, f_l, n_r, f_r

## 5. LC neuron responses

In [ ]:
lc_mask = neurons_df['type'].str.startswith('LC', na=False)
lc_types = neurons_df.loc[lc_mask, 'type'].value_counts()
print(f'LC types: {len(lc_types)}, total: {lc_mask.sum()}')

top_lc = lc_types.head(8).index.tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 6), sharex=True)
for ax, ct in zip(axes.flat, top_lc):
    il = neurons_df.loc[neurons_df['type']==ct, 'idx'].dropna().astype(int).tolist()
    if not il: ax.set_title(f'{ct} (0)'); continue
    v_m = V[0,:,il].mean(dim=1).numpy()
    v_s = V[0,:,il].std(dim=1).numpy()
    ax.plot(t_ms, v_m, 'k', lw=1.5)
    ax.fill_between(t_ms, v_m-v_s, v_m+v_s, alpha=0.2)
    for j in range(min(3, len(il))):
        ax.plot(t_ms, V[0,:,il[j]].numpy(), lw=0.5, alpha=0.4)
    ax.axvspan(T_PRE, T_PRE+T_ON, color='gold', alpha=0.15)
    ax.set_title(f'{ct} (n={len(il)})')
    ax.set_ylabel('mV')
for ax in axes[-1]: ax.set_xlabel('Time (ms)')
fig.suptitle('LC neuron flash responses (whole brain)', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
print(f'{"LC Type":>10s}  {"n":>4s}  {"FRI":>8s}')
for ct in lc_types.head(15).index:
    il = neurons_df.loc[neurons_df['type']==ct, 'idx'].dropna().astype(int).tolist()
    if not il: continue
    it = torch.tensor(il)
    print(f'{ct:>10s}  {len(il):4d}  {fri_t[it].mean():+.3f}')